# Leakage-Aware, Cross-Validated Stacking Pipeline

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv("heart.csv")
print("Unique base rows:", df.shape[0])
print("Exact duplicate rows already present:", df.duplicated().sum())

Unique base rows: 303
Exact duplicate rows already present: 1


## 1. Simulate the merge-induced duplication

In [2]:
df_merged = pd.concat([df]*3, ignore_index=True)
df_merged["patient_id"] = df_merged.index % len(df)
print("Simulated merged size:", df_merged.shape)
print("Duplicate rows in simulated merge:", df_merged.duplicated(subset=df.columns).sum())

Simulated merged size: (909, 15)
Duplicate rows in simulated merge: 607


## 2. Naive random split (leaky) vs. group-aware split (honest)

In [3]:
def eval_split(X_train, X_test, y_train, y_test, label):
    scaler = StandardScaler().fit(X_train)
    Xtr, Xte = scaler.transform(X_train), scaler.transform(X_test)
    model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE)
    model.fit(Xtr, y_train)
    preds = model.predict(Xte)
    print(f"{label:35s} Accuracy={accuracy_score(y_test, preds):.4f}  "
          f"F1={f1_score(y_test, preds):.4f}")

X = df_merged.drop(columns=["target", "patient_id"])
y = df_merged["target"]
groups = df_merged["patient_id"]

# (a) naive random split : duplicates of the same patient can appear in both sets
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
eval_split(X_tr, X_te, y_tr, y_te, "Naive random split (leaky)")

# (b) group-aware split : all copies of a patient stay on one side
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
eval_split(X.iloc[train_idx], X.iloc[test_idx], y.iloc[train_idx], y.iloc[test_idx],
           "Group-aware split (honest)")

Naive random split (leaky)          Accuracy=0.9835  F1=0.9851
Group-aware split (honest)          Accuracy=0.8361  F1=0.8438


Group-aware split (honest)          Accuracy=0.8361  F1=0.8438


**Expected pattern:** the naive split score should look inflated relative to the group-aware
split, because near/exact-duplicate patients leak between train and test. Record the actual gap
you observe here — that's direct evidence for your critical-analysis section.

## 3. Honest evaluation protocol: de-duplicated data + stratified 5-fold CV + tuning

In [4]:
X = df.drop(columns=["target"])
y = df["target"]

param_grids = {
    "Logistic Regression": (LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
                             {"C": [0.01, 0.1, 1, 10]}),
    "Decision Tree":       (DecisionTreeClassifier(random_state=RANDOM_STATE),
                             {"max_depth": [3, 5, 7, None]}),
    "Random Forest":       (RandomForestClassifier(random_state=RANDOM_STATE),
                             {"n_estimators": [100, 300], "max_depth": [5, 10, None]}),
    "XGBoost":             (XGBClassifier(eval_metric="logloss", random_state=RANDOM_STATE),
                             {"n_estimators": [100, 300], "max_depth": [3, 5]}),
    "Naive Bayes":         (GaussianNB(), {}),
    "KNN":                 (KNeighborsClassifier(), {"n_neighbors": [5, 7, 9, 11]}),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
tuned_models = {}

for name, (est, grid) in param_grids.items():
    if grid:
        search = GridSearchCV(est, grid, cv=skf, scoring="accuracy", n_jobs=-1)
        search.fit(StandardScaler().fit_transform(X), y)
        tuned_models[name] = search.best_estimator_
        print(f"{name:20s} best params: {search.best_params_}")
    else:
        tuned_models[name] = est
        print(f"{name:20s} no hyperparameters to tune")

Logistic Regression  best params: {'C': 1}
Decision Tree        best params: {'max_depth': 3}
Random Forest        best params: {'max_depth': 5, 'n_estimators': 100}
XGBoost              best params: {'max_depth': 3, 'n_estimators': 100}
Naive Bayes          no hyperparameters to tune
KNN                  best params: {'n_neighbors': 11}


Random Forest        best params: {'max_depth': 5, 'n_estimators': 100}


XGBoost              best params: {'max_depth': 3, 'n_estimators': 100}
Naive Bayes          no hyperparameters to tune
KNN                  best params: {'n_neighbors': 11}


In [5]:
from sklearn.base import clone

def cv_scores(model, X, y, cv):
    accs, precs, recs, f1s, aucs = [], [], [], [], []
    for train_idx, test_idx in cv.split(X, y):
        scaler = StandardScaler().fit(X.iloc[train_idx])
        Xtr, Xte = scaler.transform(X.iloc[train_idx]), scaler.transform(X.iloc[test_idx])
        ytr, yte = y.iloc[train_idx], y.iloc[test_idx]
        m = clone(model)
        m.fit(Xtr, ytr)
        preds = m.predict(Xte)
        probs = m.predict_proba(Xte)[:, 1]
        accs.append(accuracy_score(yte, preds))
        precs.append(precision_score(yte, preds))
        recs.append(recall_score(yte, preds))
        f1s.append(f1_score(yte, preds))
        aucs.append(roc_auc_score(yte, probs))
    return dict(Accuracy=(np.mean(accs), np.std(accs)),
                Precision=(np.mean(precs), np.std(precs)),
                Recall=(np.mean(recs), np.std(recs)),
                F1=(np.mean(f1s), np.std(f1s)),
                ROC_AUC=(np.mean(aucs), np.std(aucs)))

rows = []
for name, model in tuned_models.items():
    scores = cv_scores(model, X, y, skf)
    row = {"Model": name}
    row.update({k: f"{v[0]:.4f} ± {v[1]:.4f}" for k, v in scores.items()})
    rows.append(row)

stack = StackingClassifier(
    estimators=[(n, m) for n, m in tuned_models.items()],
    final_estimator=LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    cv=5, n_jobs=-1,
)
stack_scores = cv_scores(stack, X, y, skf)
rows.append({"Model": "Stacking (tuned, CV)",
             **{k: f"{v[0]:.4f} ± {v[1]:.4f}" for k, v in stack_scores.items()}})

cv_results_df = pd.DataFrame(rows).set_index("Model")
cv_results_df

,Accuracy,Precision,Recall,F1,ROC_AUC
Model,,,,,
Logistic Regression,0.8416 ± 0.0448,0.8195 ± 0.0538,0.9152 ± 0.0227,0.8638 ± 0.0345,0.8901 ± 0.0469
Decision Tree,0.8052 ± 0.0473,0.8135 ± 0.0636,0.8424 ± 0.0353,0.8259 ± 0.0366,0.8166 ± 0.0525
Random Forest,0.8251 ± 0.0669,0.8142 ± 0.0788,0.8909 ± 0.0242,0.8494 ± 0.0512,0.9074 ± 0.0523
XGBoost,0.8120 ± 0.0490,0.8128 ± 0.0694,0.8606 ± 0.0454,0.8338 ± 0.0400,0.8773 ± 0.0433
Naive Bayes,0.8084 ± 0.0697,0.8247 ± 0.0992,0.8424 ± 0.0402,0.8300 ± 0.0543,0.8875 ± 0.0538
KNN,0.8083 ± 0.0749,0.7905 ± 0.0914,0.9030 ± 0.0297,0.8401 ± 0.0544,0.8797 ± 0.0650
"Stacking (tuned, CV)",0.8218 ± 0.0479,0.8083 ± 0.0613,0.8909 ± 0.0309,0.8460 ± 0.0357,0.9065 ± 0.0503
